# Classificadores Multiclasse Baseados em Algoritmos de Agrupamento

Implementação em Python para os datasets Adult e Dry Bean.

In [1]:
# Importação de bibliotecas
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
# Instale scikit-fuzzy se necessário
# !pip install scikit-fuzzy

import skfuzzy as fuzz

ModuleNotFoundError: No module named 'seaborn'

## Função para encontrar o número ideal de clusters (Elbow Method)

In [ ]:
def elbow_method(X, max_k=10):
    distortions = []
    for k in range(1, max_k+1):
        kmeans = KMeans(n_clusters=k, random_state=0)
        kmeans.fit(X)
        distortions.append(kmeans.inertia_)
    plt.plot(range(1, max_k+1), distortions, marker='o')
    plt.xlabel('Número de clusters')
    plt.ylabel('Distortion')
    plt.title('Método do Cotovelo')
    plt.show()
    # Retorne o valor de k sugerido visualmente
    return np.argmin(np.diff(distortions, 2)) + 2

## Função para treinar e avaliar classificadores baseados em agrupamento

In [ ]:
def cluster_classifier(X_train, y_train, X_test, y_test, clusterer):
    # Ajusta o clusterer nos dados de treino
    clusterer.fit(X_train)
    # Associa cada cluster à classe majoritária
    clusters = clusterer.labels_ if hasattr(clusterer, 'labels_') else clusterer.predict(X_train)
    cluster_to_class = {}
    for c in np.unique(clusters):
        mask = clusters == c
        majority_class = pd.Series(y_train[mask]).mode()[0]
        cluster_to_class[c] = majority_class
    # Predição nos dados de teste
    test_clusters = clusterer.predict(X_test)
    y_pred = np.array([cluster_to_class.get(c, -1) for c in test_clusters])
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    return acc, cm

## Função para rodar experimentos repetidos

In [ ]:
def run_experiments(X, y, clusterer_class, n_clusters, n_runs=30):
    accs = []
    cms = []
    for seed in range(1, n_runs+1):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y)
        clusterer = clusterer_class(n_clusters=n_clusters, random_state=seed)
        acc, cm = cluster_classifier(X_train, y_train, X_test, y_test, clusterer)
        accs.append(acc)
        cms.append(cm)
    return np.array(accs), cms

## Carregamento e preparação dos dados (Exemplo: Adult)
Repita para Dry Bean conforme necessário.

In [ ]:
# Substitua pelo caminho correto do arquivo Adult
adult = pd.read_csv('adult.csv')
# Pré-processamento
adult = adult.dropna()
for col in adult.select_dtypes(include='object').columns:
    adult[col] = LabelEncoder().fit_transform(adult[col])
X_adult = adult.drop('class', axis=1).values
y_adult = adult['class'].values
X_adult = StandardScaler().fit_transform(X_adult)

## Encontrar número ideal de clusters para Adult

In [ ]:
k_adult = elbow_method(X_adult, max_k=10)
print(f'Número ideal de clusters (Adult): {k_adult}')

## Executar experimentos com três classificadores de agrupamento

In [ ]:
# KMeans
accs_kmeans, cms_kmeans = run_experiments(X_adult, y_adult, KMeans, k_adult)
# Agglomerative
accs_agg, cms_agg = run_experiments(X_adult, y_adult, AgglomerativeClustering, k_adult)
# Spectral
accs_spec, cms_spec = run_experiments(X_adult, y_adult, SpectralClustering, k_adult)
print('KMeans:', accs_kmeans.mean(), accs_kmeans.std())
print('Agglomerative:', accs_agg.mean(), accs_agg.std())
print('Spectral:', accs_spec.mean(), accs_spec.std())

## Exibir matriz de confusão média

In [ ]:
def plot_mean_cm(cms, labels):
    mean_cm = np.mean(cms, axis=0)
    plt.figure(figsize=(6,5))
    sns.heatmap(mean_cm, annot=True, fmt='.0f', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predito')
    plt.ylabel('Verdadeiro')
    plt.title('Matriz de Confusão Média')
    plt.show()

# Exemplo para KMeans
plot_mean_cm(cms_kmeans, np.unique(y_adult))

## Repita o mesmo processo para o dataset Dry Bean
### (Carregue, pré-processe, encontre k, execute experimentos e avalie)

In [ ]:
# Substitua pelo caminho correto do arquivo Dry Bean
dry_beans = pd.read_csv('dry_beans.csv')
# Pré-processamento
dry_beans = dry_beans.dropna()
for col in dry_beans.select_dtypes(include='object').columns:
    dry_beans[col] = LabelEncoder().fit_transform(dry_beans[col])
X_dry_beans = dry_beans.drop('Class', axis=1).values
y_dry_beans = dry_beans['Class'].values
X_dry_beans = StandardScaler().fit_transform(X_dry_beans)

## Encontrar número ideal de clusters para Dry Bean

In [ ]:
k_dry_beans = elbow_method(X_dry_beans, max_k=10)
print(f'Número ideal de clusters (Dry Bean): {k_dry_beans}')

## Executar experimentos com três classificadores de agrupamento para Dry Bean

In [ ]:
# KMeans
accs_kmeans_dry, cms_kmeans_dry = run_experiments(X_dry_beans, y_dry_beans, KMeans, k_dry_beans)
# Agglomerative
accs_agg_dry, cms_agg_dry = run_experiments(X_dry_beans, y_dry_beans, AgglomerativeClustering, k_dry_beans)
# Spectral
accs_spec_dry, cms_spec_dry = run_experiments(X_dry_beans, y_dry_beans, SpectralClustering, k_dry_beans)
print('KMeans (Dry Bean):', accs_kmeans_dry.mean(), accs_kmeans_dry.std())
print('Agglomerative (Dry Bean):', accs_agg_dry.mean(), accs_agg_dry.std())
print('Spectral (Dry Bean):', accs_spec_dry.mean(), accs_spec_dry.std())

## Exibir matriz de confusão média para Dry Bean

In [ ]:
# Exemplo para KMeans em Dry Bean
plot_mean_cm(cms_kmeans_dry, np.unique(y_dry_beans))

In [ ]:
# Função para treinar e avaliar Fuzzy K-Means (c-means)
def fuzzy_cmeans_classifier(X_train, y_train, X_test, y_test, n_clusters, m=2.0, error=1e-5, maxiter=1000):
    # Transpor para formato esperado pelo cmeans
    cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
        X_train.T, n_clusters, m, error=error, maxiter=maxiter, seed=0)
    # Para cada cluster, associa à classe majoritária
    cluster_labels = np.argmax(u, axis=0)
    cluster_to_class = {}
    for c in range(n_clusters):
        mask = cluster_labels == c
        if np.any(mask):
            majority_class = pd.Series(y_train[mask]).mode()[0]
            cluster_to_class[c] = majority_class
        else:
            cluster_to_class[c] = -1  # fallback
    # Predição nos dados de teste
    u_test, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
        X_test.T, cntr, m, error=error, maxiter=maxiter)
    test_clusters = np.argmax(u_test, axis=0)
    y_pred = np.array([cluster_to_class.get(c, -1) for c in test_clusters])
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    return acc, cm

# Função para rodar experimentos com fuzzy c-means
def run_fuzzy_experiments(X, y, n_clusters, n_runs=30):
    accs = []
    cms = []
    for seed in range(1, n_runs+1):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y)
        acc, cm = fuzzy_cmeans_classifier(X_train, y_train, X_test, y_test, n_clusters)
        accs.append(acc)
        cms.append(cm)
    return np.array(accs), cms

## Executar experimentos com Fuzzy K-Means (c-means) para Adult

In [ ]:
# Executar experimentos com Fuzzy K-Means (c-means)
accs_fuzzy, cms_fuzzy = run_fuzzy_experiments(X_adult, y_adult, k_adult)
print('Fuzzy K-Means:', accs_fuzzy.mean(), accs_fuzzy.std())

# Exibir matriz de confusão média para Fuzzy K-Means
plot_mean_cm(cms_fuzzy, np.unique(y_adult))

## Executar experimentos com Fuzzy K-Means (c-means) para Dry Bean

In [ ]:
# Executar experimentos com Fuzzy K-Means (c-means) para Dry Bean
accs_fuzzy_dry, cms_fuzzy_dry = run_fuzzy_experiments(X_dry_beans, y_dry_beans, k_dry_beans)
print('Fuzzy K-Means (Dry Bean):', accs_fuzzy_dry.mean(), accs_fuzzy_dry.std())

# Exibir matriz de confusão média para Fuzzy K-Means em Dry Bean
plot_mean_cm(cms_fuzzy_dry, np.unique(y_dry_beans))

## Base de Dados Dry Bean

A base de dados **Dry Bean** pode ser obtida gratuitamente no repositório UCI Machine Learning Repository:

- [Dry Bean Dataset - UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Dry+Bean+Dataset)

O arquivo principal é `Dry_Bean_Dataset.xlsx`.

## Como usar a base Dry Bean no Python

1. Baixe o arquivo `Dry_Bean_Dataset.xlsx` do [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Dry+Bean+Dataset).
2. Coloque o arquivo na mesma pasta do seu notebook ou forneça o caminho correto.
3. Use o pandas para carregar e preparar os dados:

```python
import pandas as pd

# Carregar o arquivo Excel
df = pd.read_excel('Dry_Bean_Dataset.xlsx')

# Visualizar as primeiras linhas
print(df.head())

# Separar atributos e rótulos
X = df.drop('Class', axis=1).values
y = df['Class'].values
```

4. Faça o pré-processamento necessário (normalização, codificação, etc.) antes de usar nos classificadores.